# 118 — Presupuestos de pasos, tokens, costo y tiempo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Un agente tiene costo ABIERTO: cada iteración re-paga el contexto acumulado. Se
presupuestan **cuatro monedas por separado** — pasos, tokens, dinero, tiempo — y la
primera que se agote detiene la tarea (no son intercambiables).

**Modelo de costo** (bucle de n pasos, contexto inicial c0, Δ tokens nuevos por paso,
s tokens de salida por paso):

```text
entrada ≈ n·c0 + Δ·n·(n-1)/2     ← CUADRÁTICO en n si no se compacta
salida  ≈ n·s                     ← lineal
costo   = entrada·p_in + salida·p_out   (p_out suele ser varias veces p_in)
```

Consecuencia: duplicar los pasos casi cuadruplica los tokens de entrada. Presupuesto
de tokens y gestión de contexto (115) son la misma batalla.

### 📉 Contrato de tres fases

1. **Estimar (antes):** presupuesto por sub-tarea del plan (112) + reserva (~20 %).
2. **Medir (durante):** telemetría por paso — spans con tokens, costo y estado;
   alertas al 80 % de cualquier moneda, ANTES del corte.
3. **Actuar (al agotarse):** el presupuesto se comprueba ANTES de cada acción; parada
   limpia con checkpoint (115) + estado parcial + reporte de qué falta.

Los reintentos (113) consumen del mismo pozo y son señal diagnóstica, no ruido. El
laboratorio `observability` emite la fase "medir" mínima: 3 spans con tokens
(120 + 80 + 40 = 240) y duración — los datos sobre los que se corta o alerta.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=118)
show(result)


## Reflexión

1. ¿Por qué "comprobar el presupuesto después de actuar" es un error de diseño y qué
   relación tiene con los puntos consistentes de checkpoint de la clase 115?
2. El laboratorio muestra tokens por span pero no costo en dinero. ¿Qué dos datos
   externos necesitas para convertir spans en factura, y por qué deben versionarse?
3. Tu agente se detuvo por presupuesto al 100 % de tokens con 40 % de los pasos
   usados. ¿Qué diagnóstico sugiere esa asimetría y qué arreglo de la clase 115
   probarías primero?